# 03 - Download de PDFs

Baixa os PDFs dos papers classificados como `relevant` no notebook 02, a partir do `oa_pdf_url`.

**Saída:**
- Arquivos PDF em `pdfs/`.
- `data/processed/papers_with_pdf.parquet` - base do notebook 02 mais colunas `pdf_status`, `pdf_local_path`, `pdf_size_bytes`, `pdf_http_status`, `pdf_error`.

In [1]:
import os
import re
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

c:\Users\fredb\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PDF_DIR = PROJECT_DIR / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

USER_EMAIL = ""
N_WORKERS = 6
MAX_SIZE_MB = 100
TIMEOUT_SECONDS = 60
INCLUDE_UNCERTAIN = False

INPUT_PATH = PROCESSED_DIR / "papers_classified.parquet"
LOG_PATH = PROCESSED_DIR / "download_log.jsonl"
FINAL_PATH = PROCESSED_DIR / "papers_with_pdf.parquet"

print(f"PDF dir:    {PDF_DIR}")
print(f"Input:      {INPUT_PATH}")
print(f"Log:        {LOG_PATH}")
print(f"Saída:      {FINAL_PATH}")
print(f"Workers:    {N_WORKERS}, max size: {MAX_SIZE_MB} MB, timeout: {TIMEOUT_SECONDS}s")

PDF dir:    c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\pdfs
Input:      c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_classified.parquet
Log:        c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\download_log.jsonl
Saída:      c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_with_pdf.parquet
Workers:    6, max size: 100 MB, timeout: 60s


In [4]:
df = pd.read_parquet(INPUT_PATH)
print(f"Total classificados: {len(df)}")
print(f"\nPor label:")
print(df["label"].value_counts(dropna=False))

labels_to_download = ["relevant"]
if INCLUDE_UNCERTAIN:
    labels_to_download.append("uncertain")

df_dl = df[df["label"].isin(labels_to_download)].copy().reset_index(drop=True)
has_url = df_dl["oa_pdf_url"].notna() & (df_dl["oa_pdf_url"] != "")
print(f"\nPara baixar: {len(df_dl)} (labels: {labels_to_download})")
print(f"  Com URL: {int(has_url.sum())}")
print(f"  Sem URL: {int((~has_url).sum())}")

Total classificados: 1405

Por label:
label
relevant        1078
not_relevant     172
uncertain        155
Name: count, dtype: int64

Para baixar: 1078 (labels: ['relevant'])
  Com URL: 1072
  Sem URL: 6


## Helpers

- **`sanitize_filename`**: `paper_id` contém `:` (ex: `arxiv:2104.01234`) e às vezes `/` — caracteres inválidos pra nome de arquivo no Windows. Substitui por `_`.
- **`build_session`**: `requests.Session` com User-Agent polite (com email) e `Retry` adapter automático em 429/5xx (3 tentativas, backoff 2× exponencial).

In [5]:
INVALID_CHARS = re.compile(r'[<>:"/\\|?*]')

def sanitize_filename(paper_id: str) -> str:
    return INVALID_CHARS.sub("_", paper_id)

def build_session() -> requests.Session:
    retry = Retry(
        total=3,
        backoff_factor=2.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "HEAD"],
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s = requests.Session()
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": f"causal-ml-project ({USER_EMAIL})"})
    return s

In [6]:
def download_pdf(paper_id: str, pdf_url: str, session: requests.Session,
                  pdf_dir: Path = PDF_DIR, max_size_mb: int = MAX_SIZE_MB,
                  timeout: int = TIMEOUT_SECONDS) -> dict:
    base = {
        "paper_id": paper_id,
        "url": pdf_url,
        "status": "failed",
        "local_path": None,
        "size_bytes": None,
        "http_status": None,
        "error": None,
    }

    if not pdf_url or (isinstance(pdf_url, float) and pd.isna(pdf_url)):
        base["status"] = "no_url"
        return base

    filename = sanitize_filename(paper_id) + ".pdf"
    local_path = pdf_dir / filename

    if local_path.exists() and local_path.stat().st_size > 0:
        base["status"] = "already_exists"
        base["local_path"] = str(local_path)
        base["size_bytes"] = local_path.stat().st_size
        return base

    tmp_path = local_path.with_suffix(".tmp")

    try:
        with session.get(pdf_url, stream=True, timeout=timeout, allow_redirects=True) as r:
            base["http_status"] = r.status_code
            r.raise_for_status()

            ct = r.headers.get("Content-Type", "").lower()
            if "html" in ct and "pdf" not in ct:
                base["error"] = f"got HTML (content-type={ct[:80]})"
                return base

            cl = r.headers.get("Content-Length")
            if cl and int(cl) > max_size_mb * 1024 * 1024:
                base["error"] = f"too large ({int(cl)/1024/1024:.1f} MB by header)"
                return base

            size = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=64 * 1024):
                    if chunk:
                        f.write(chunk)
                        size += len(chunk)
                        if size > max_size_mb * 1024 * 1024:
                            base["error"] = f"size exceeded mid-stream ({size/1024/1024:.1f} MB)"
                            return base

        with open(tmp_path, "rb") as f:
            magic = f.read(5)
        if not magic.startswith(b"%PDF"):
            base["error"] = f"not a PDF (magic={magic!r})"
            return base

        tmp_path.rename(local_path)
        base["status"] = "success"
        base["local_path"] = str(local_path)
        base["size_bytes"] = size
        return base

    except requests.exceptions.HTTPError as e:
        base["http_status"] = e.response.status_code if e.response is not None else None
        base["error"] = f"HTTPError {base['http_status']}"
    except requests.exceptions.Timeout:
        base["error"] = "timeout"
    except requests.exceptions.ConnectionError as e:
        base["error"] = f"ConnectionError: {str(e)[:200]}"
    except requests.exceptions.RequestException as e:
        base["error"] = f"{type(e).__name__}: {str(e)[:200]}"
    except Exception as e:
        base["error"] = f"{type(e).__name__}: {str(e)[:200]}"
    finally:
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except Exception:
                pass

    return base

## Run com checkpoint

In [7]:
def get_done_paper_ids(pdf_dir: Path, paper_ids) -> set:
    done = set()
    for pid in paper_ids:
        fn = sanitize_filename(pid) + ".pdf"
        p = pdf_dir / fn
        if p.exists() and p.stat().st_size > 0:
            done.add(pid)
    return done


def run_downloads(df: pd.DataFrame, log_path: Path, pdf_dir: Path = PDF_DIR,
                   n_workers: int = N_WORKERS) -> int:
    done_ids = get_done_paper_ids(pdf_dir, df["paper_id"])
    todo = df[~df["paper_id"].isin(done_ids)].copy()
    print(f"Total: {len(df):>5}")
    print(f"Já baixados (PDF local): {len(done_ids):>5}")
    print(f"A baixar (inclui falhas anteriores): {len(todo):>5}\n")

    if len(todo) == 0:
        return 0

    session = build_session()
    start = time.time()
    n_success = 0
    n_failed = 0
    n_no_url = 0

    with open(log_path, "a", encoding="utf-8") as f_log:
        with ThreadPoolExecutor(max_workers=n_workers) as pool:
            futures = {
                pool.submit(download_pdf, row.paper_id, row.oa_pdf_url, session): row.paper_id
                for row in todo.itertuples()
            }
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Baixando"):
                try:
                    result = fut.result()
                except Exception as e:
                    result = {
                        "paper_id": futures[fut],
                        "url": None,
                        "status": "failed",
                        "local_path": None,
                        "size_bytes": None,
                        "http_status": None,
                        "error": f"future_exception: {type(e).__name__}: {e}",
                    }
                f_log.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_log.flush()

                status = result["status"]
                if status in ("success", "already_exists"):
                    n_success += 1
                elif status == "no_url":
                    n_no_url += 1
                else:
                    n_failed += 1

    elapsed = time.time() - start
    print(f"\nProcessados: {n_success + n_failed + n_no_url}")
    print(f"  Sucesso:   {n_success}")
    print(f"  Sem URL:   {n_no_url}")
    print(f"  Falha:     {n_failed}")
    pdfs_on_disk = list(pdf_dir.glob("*.pdf"))
    total_mb = sum(p.stat().st_size for p in pdfs_on_disk) / 1024 / 1024
    print(f"  Total no disco: {total_mb:.1f} MB ({len(pdfs_on_disk)} arquivos)")
    print(f"  Tempo:     {elapsed:.1f}s ({(n_success + n_failed + n_no_url)/max(elapsed,1):.2f} req/s)")
    return n_success + n_failed + n_no_url

In [9]:
n = run_downloads(df_dl, LOG_PATH)

Total:  1078
Já baixados (PDF local):   602
A baixar (inclui falhas anteriores):   476



Baixando: 100%|██████████| 476/476 [05:13<00:00,  1.52it/s]


Processados: 476
  Sucesso:   0
  Sem URL:   6
  Falha:     470
  Total no disco: 1116.7 MB (602 arquivos)
  Tempo:     314.0s (1.52 req/s)


## Salvar resultado final

Carrega o log, dedupica por `paper_id` (preferindo sucesso sobre falha — mesma lógica do notebook 02) e faz merge com `papers_classified.parquet`. Papers com `label != "relevant"` (não tentados) ganham `pdf_status = "not_attempted"`.

In [10]:
def load_log(log_path: Path) -> pd.DataFrame:
    rows = []
    if not log_path.exists():
        return pd.DataFrame(rows)
    with open(log_path, encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return pd.DataFrame(rows)


df_log = load_log(LOG_PATH)

if len(df_log) > 0:
    df_log["_is_success"] = df_log["status"].isin(["success", "already_exists"])
    df_log["_order"] = range(len(df_log))
    df_log = df_log.sort_values(["paper_id", "_is_success", "_order"],
                                 ascending=[True, True, True])
    df_log = df_log.drop_duplicates("paper_id", keep="last").reset_index(drop=True)
    df_log = df_log.drop(columns=["_is_success", "_order", "url"])
    df_log = df_log.rename(columns={
        "status": "pdf_status",
        "local_path": "pdf_local_path",
        "size_bytes": "pdf_size_bytes",
        "http_status": "pdf_http_status",
        "error": "pdf_error",
    })

log_pids = set(df_log["paper_id"]) if len(df_log) > 0 else set()
extras = []
for pid in df_dl["paper_id"]:
    if pid in log_pids:
        continue
    fn = sanitize_filename(pid) + ".pdf"
    p = PDF_DIR / fn
    if p.exists() and p.stat().st_size > 0:
        extras.append({
            "paper_id": pid,
            "pdf_status": "already_exists",
            "pdf_local_path": str(p),
            "pdf_size_bytes": p.stat().st_size,
            "pdf_http_status": None,
            "pdf_error": None,
        })
if extras:
    df_log = pd.concat([df_log, pd.DataFrame(extras)], ignore_index=True)

df_final = df.merge(df_log, on="paper_id", how="left")
df_final["pdf_status"] = df_final["pdf_status"].fillna("not_attempted")
df_final.to_parquet(FINAL_PATH, index=False)
print(f"Salvo em {FINAL_PATH}")
print(f"Linhas: {len(df_final)}")

Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_with_pdf.parquet
Linhas: 1405


## Resumo

In [11]:
df_attempted = df_final[df_final["label"].isin(labels_to_download)]

print("Status por label (papers tentados):")
print(pd.crosstab(df_attempted["label"], df_attempted["pdf_status"], margins=True))

success_mask = df_attempted["pdf_status"].isin(["success", "already_exists"])
n_success = int(success_mask.sum())
print(f"\nSucesso: {n_success} / {len(df_attempted)} ({n_success/max(len(df_attempted),1)*100:.1f}%)")

errors = df_attempted[df_attempted["pdf_error"].notna()]
if len(errors) > 0:
    print(f"\nTop categorias de erro ({len(errors)} total):")
    err_cat = errors["pdf_error"].astype(str).str.split(":").str[0]
    print(err_cat.value_counts().head(10))

    print(f"\nExemplos de erros (3 amostras):")
    for row in errors.sample(n=min(3, len(errors)), random_state=1).itertuples():
        print(f"  [{row.rank}] {row.title[:80]}")
        print(f"    URL: {(row.oa_pdf_url or '')[:100]}")
        print(f"    Err: {row.pdf_error}")

Status por label (papers tentados):
pdf_status  failed  no_url  success   All
label                                    
relevant       470       6      602  1078
All            470       6      602  1078

Sucesso: 602 / 1078 (55.8%)

Top categorias de erro (470 total):
pdf_error
HTTPError 403                                       205
got HTML (content-type=text/html; charset=utf-8)    108
got HTML (content-type=text/html;charset=utf-8)      93
HTTPError 404                                        27
ConnectionError                                      18
timeout                                               7
got HTML (content-type=text/html)                     6
not a PDF (magic=b'Verif')                            1
InvalidURL                                            1
not a PDF (magic=b'<scri')                            1
Name: count, dtype: int64

Exemplos de erros (3 amostras):
  [1303] SMOTE-LOF for noise identification in imbalanced data classification
    URL: https://doi.or